# SmartHeart Colab Training

## 1. Clone the repository

Clone the latest `main` branch into the Colab workspace. If it is already present, update it with a fast-forward pull, then make it the active working directory.

In [1]:
!if [ -d /content/smart-heart/.git ]; then git -C /content/smart-heart pull --ff-only origin main; else git clone --branch main --single-branch https://github.com/Bojan-Ivanovski/smart-heart.git /content/smart-heart; fi
%cd /content/smart-heart


Cloning into '/content/smart-heart'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 78 (delta 2), reused 6 (delta 1), pack-reused 69 (from 1)
Receiving objects: 100% (78/78), 49.46 MiB | 19.41 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/smart-heart


## 2. Configure the run

Select matching hardware from **Runtime > Change runtime type**, then configure the accelerator, dataset, model, windowing, and training values below. Use `cuda` for GPU, `xla` for TPU, or `cpu` when no accelerator is enabled.

In [2]:
import os

DEVICE = "cuda"  # @param ["cuda", "xla", "cpu"]
DATASET = "gameplay"  # @param ["gameplay", "children"]
MODEL_ID = "google/gemma-3-270m"  # @param {type:"string"}
EPOCHS = 1  # @param {type:"integer"}
BATCH_SIZE = 1  # @param {type:"integer"}
WINDOW_SIZE = 4096  # @param {type:"integer"}
WINDOW_STRIDE = 4096  # @param {type:"integer"}
LEARNING_RATE = 0.0001  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.environ["SMARTHEART_DEVICE"] = DEVICE
print(f"device={DEVICE}, dataset={DATASET}, model={MODEL_ID}")


device=cuda, dataset=gameplay, model=google/gemma-3-270m


## 3. Install dependencies

Upgrade `pip` and install the repository's pinned environment. CUDA and CPU runs use the base requirements; XLA runs additionally install the TPU runtime and its official wheel source.

In [3]:
%%bash
set -euo pipefail
python -m pip install --upgrade pip
if [[ "${SMARTHEART_DEVICE}" == "xla" ]]; then
    python -m pip install -r requirements-tpu.txt
else
    python -m pip install -r requirements.txt
fi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/StanfordBDHG/OpenTSLM.git (to revision refs/pull/47/head) to /tmp/pip-install-bl90ehh9/opentslm_858219c959a14ff1a83e031a8c1e08c2
  Did not find branch or tag 'refs/pull/47/head', assuming revision or ref.
  Resolved https://github.com/StanfordBDHG/OpenTSLM.git to commit 3356bbd4ecd2a88f713784d3e14f71a2b5b5b2f2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other 

  Running command git clone --filter=blob:none --quiet https://github.com/StanfordBDHG/OpenTSLM.git /tmp/pip-install-bl90ehh9/opentslm_858219c959a14ff1a83e031a8c1e08c2
  Running command git fetch -q https://github.com/StanfordBDHG/OpenTSLM.git refs/pull/47/head
  Running command git checkout -q 3356bbd4ecd2a88f713784d3e14f71a2b5b5b2f2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatibl

## 4. Extract the datasets

Verify both tracked ZIP archives, extract them into the directory layout expected by SmartHeart, and report the discovered recording counts. Re-running this cell refreshes the extracted files.

In [4]:
%%bash
set -euo pipefail
test -f datasets/adhd_children_dataset.zip
test -f datasets/adhd_individuals_gameplay_dataset.zip
unzip -q -o datasets/adhd_children_dataset.zip -d datasets
unzip -q -o datasets/adhd_individuals_gameplay_dataset.zip -d datasets
test -d datasets/adhd_children_dataset
test -d datasets/adhd_individuals_gameplay_dataset
echo "Children MAT files: $(find datasets/adhd_children_dataset -type f -name '*.mat' | wc -l)"
echo "Gameplay CSV files: $(find datasets/adhd_individuals_gameplay_dataset -type f -name '*.csv' | wc -l)"


Children MAT files: 121
Gameplay CSV files: 150


## 5. Validate the environment and preview data

Report the installed PyTorch version and CUDA availability, then preview the selected dataset to confirm discovery and loading before model training begins.

In [5]:
!python -c "import torch; print('torch:', torch.__version__); print('cuda_available:', torch.cuda.is_available())"
!python -m src.main --mode preview --dataset {DATASET} --window-size {WINDOW_SIZE} --window-stride {WINDOW_STRIDE}


torch: 2.9.0+cu128
cuda_available: True
2026-08-30 19:55:43.026777: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-30 19:55:43.097690: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
dataset: gameplay
samples: 100
answer: YES
time_series_shape: (5, 1225)
prompt: Does this patient have ADHD? Window covers timesteps 0 to 1225.


## 6. Authenticate with Hugging Face

Gemma access may require accepting its license on Hugging Face. Add a Colab secret named `HF_TOKEN` using the key icon in the left sidebar and grant this notebook access. The next cell stores it in Hugging Face's local cache without printing it.

In [7]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HUGFACE")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("HF_TOKEN is not configured. Public models can still be downloaded.")


HF_TOKEN is not configured. Public models can still be downloaded.


## 7. Train the model

Set the TPU PJRT environment when XLA is selected, then start the SmartHeart training CLI with the values configured above. LoRA remains enabled by the application's default configuration.

In [ ]:
if DEVICE == "xla":
    os.environ["PJRT_DEVICE"] = "TPU"

!python -m src.main --mode train --dataset {DATASET} --device {DEVICE} --model-id {MODEL_ID} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --window-size {WINDOW_SIZE} --window-stride {WINDOW_STRIDE} --learning-rate {LEARNING_RATE} --seed {SEED}


## 8. Inspect checkpoints

List every checkpoint artifact produced by training together with its allocated file size.

In [ ]:
!find checkpoints -type f -printf '%p (%k KB)\n'
